# 09B – Batch Prediction Pipeline (Enterprise)

Generate bankruptcy predictions for multiple companies using the finalized production model.

## Business Objective
Build a production-style batch inference pipeline capable of scoring multiple companies at once and exporting the results.

In [ ]:
import pandas as pd
import joblib

MODEL_PATH='production_bankruptcy_model.joblib'
INPUT_DATA='american_bankruptcy_cleaned.csv'

model=joblib.load(MODEL_PATH)
df=pd.read_csv(INPUT_DATA)


In [ ]:
target='status_label' if 'status_label' in df.columns else 'target'

if target in df.columns:
    X=df.drop(columns=[target])
else:
    X=df.copy()

predictions=model.predict(X)
probabilities=model.predict_proba(X)[:,1]


In [ ]:
results=X.copy()

results['Prediction']=predictions
results['Bankruptcy_Probability']=probabilities

results['Prediction_Label']=results['Prediction'].map({
    0:'Healthy',
    1:'Bankrupt'
})

results.to_csv('batch_prediction_results.csv',index=False)

results.head()


In [ ]:
summary=pd.DataFrame({
    'Metric':[
        'Total Records',
        'Healthy Predictions',
        'Bankrupt Predictions',
        'Average Bankruptcy Probability'
    ],
    'Value':[
        len(results),
        (results['Prediction']==0).sum(),
        (results['Prediction']==1).sum(),
        round(results['Bankruptcy_Probability'].mean(),4)
    ]
})

summary.to_csv('batch_prediction_summary.csv',index=False)

summary


## Deliverables

- `batch_prediction_results.csv`
- `batch_prediction_summary.csv`

These files can be delivered to business users, analysts, or downstream systems for decision-making and reporting.

## Executive Summary

This notebook demonstrates a production-style batch scoring pipeline. It loads the serialized model, scores every company in the cleaned dataset, generates bankruptcy probabilities, assigns prediction labels, and exports results for operational use.